# 模型效率统计汇总

展示各个 VSR 模型的效率指标：参数量、模型大小、训练时间、推理时间

**说明**:
- 在下方 config cell 中选择数据集（`RB` / `KF256` / `all`）
- Batch Size 为 frame-level（非 sequence-level）
- Infer Time/Seq 为完整序列的推理时间
- Epochs 为实际运行的 epochs（从训练日志获取，考虑了早停机制）

In [1]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# 配置：选择数据集
# ══════════════════════════════════════════════════════════════════════════════
# 可选: 'RB', 'KF256', 'all'（展示所有数据集，每个数据集一张表）
# DATASET = 'KF256'
DATASET = 'ShallowWater'

stats_dir = Path('/data/yc/Fluid_VSR/efficiency_stats')
# ══════════════════════════════════════════════════════════════════════════════

# 自动发现所有数据集
all_jsons = sorted(stats_dir.glob('*.json'))
all_jsons = [f for f in all_jsons if f.stem != 'summary' and not f.stem.startswith('summary')]

datasets_found = sorted(set(f.stem.rsplit('_', 1)[-1] for f in all_jsons))
print(f'效率统计目录: {stats_dir}')
print(f'发现数据集: {datasets_found}')

if DATASET == 'all':
    datasets_to_show = datasets_found
else:
    datasets_to_show = [DATASET]
print(f'将展示: {datasets_to_show}')

效率统计目录: /data/yc/Fluid_VSR/efficiency_stats
发现数据集: ['KF256', 'RB', 'ShallowWater']
将展示: ['ShallowWater']


In [3]:
def load_dataset_stats(stats_dir, dataset_name):
    """加载某个数据集的所有效率 JSON，返回 DataFrame"""
    stats_files = sorted(stats_dir.glob(f'*_{dataset_name}.json'))
    records = []
    for f in stats_files:
        with open(f, 'r') as fp:
            stats = json.load(fp)
        record = {
            "Model": stats["model_name"],
            "Params (M)": f"{stats['params_total']/1e6:.2f}" if stats.get('params_total') else "-",
            "Size (MB)": f"{stats['model_size_mb']:.1f}" if stats.get('model_size_mb') else "-",
            "Batch Size": stats['training'].get('batch_size', '-') or '-',
            "Epochs": stats['training'].get('total_epochs', '-') or '-',
            "Train Iters": stats['training'].get('total_iters', '-') or '-',
            "Train Time (h)": f"{stats['training']['total_time_sec']/3600:.2f}" if stats['training'].get('total_time_sec') else "-",
            "Time/Iter (s)": f"{stats['training']['time_per_iter_sec']:.3f}" if stats['training'].get('time_per_iter_sec') else "-",
            "Infer Time/Seq (s)": f"{stats['inference']['time_per_sequence_sec']:.4f}" if stats['inference'].get('time_per_sequence_sec') else "-",
            "Infer Mem (GB)": f"{stats['inference']['peak_memory_mb']/1024:.2f}" if stats['inference'].get('peak_memory_mb') else "-",
        }
        records.append(record)
    df = pd.DataFrame(records).sort_values("Model").reset_index(drop=True)
    return df

# 加载并展示
all_dfs = {}
for ds in datasets_to_show:
    df = load_dataset_stats(stats_dir, ds)
    all_dfs[ds] = df
    print(f'\n{"="*60}')
    print(f'  {ds} 数据集 — {len(df)} 个模型')
    print(f'{"="*60}')
    display(df)


  ShallowWater 数据集 — 5 个模型


,Model,Params (M),Size (MB),Batch Size,Epochs,Train Iters,Train Time (h),Time/Iter (s),Infer Time/Seq (s),Infer Mem (GB)
0,EDSR,1.52,5.8,320,100,31.0,0.16,-,0.0128,2.93
1,FNO2d,0.69,5.2,300,95,47.5,0.25,-,0.0209,6.03
2,SDIFT,3.92,15.2,-,-,-,-,-,21.4913,1.38
3,SRNO,2.01,7.7,32,105,427.35,2.89,-,0.1878,5.83
4,SwinIR,3.84,18.6,96,60,110.4,0.70,-,0.0849,0.97


## 导出 CSV / LaTeX

In [4]:
# for ds, df in all_dfs.items():
#     csv_path = stats_dir / f'summary_{ds}.csv'
#     tex_path = stats_dir / f'summary_{ds}.tex'
#     df.to_csv(csv_path, index=False)
#     df.to_latex(tex_path, index=False)
#     print(f'[{ds}] CSV: {csv_path}')
#     print(f'[{ds}] LaTeX: {tex_path}')

In [5]:
# # 导出 CSV
# output_csv = '/data/yc/Fluid_VSR/efficiency_stats/summary_RB.csv'
# df.to_csv(output_csv, index=False)
# print(f"已保存到: {output_csv}")

# # 导出 LaTeX
# output_tex = '/data/yc/Fluid_VSR/efficiency_stats/summary_RB.tex'
# df.to_latex(output_tex, index=False)
# print(f"已保存到: {output_tex}")